# 01 — Data Validation

This notebook quantifies data coverage from the gathering phase *before* any modeling.
It answers six questions that determine whether we have enough usable data to proceed:

1. **Market population** — how many resolved markets, by type and season?
2. **Classifier health** — does the new market_type distribution look right after the fix?
3. **Price coverage** — how many markets have a usable pre-game price snapshot + open→close delta?
4. **Match rate** — how well do moneyline + spread markets join to NFL games?
5. **Feature coverage** — Elo, travel, and sportsbook lines on matched markets?
6. **Calibration sanity** — does the Polymarket implied probability track realized outcomes?

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from abcm import config

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

print('Data dir:', config.DATA_DIR)
print('Years:', config.YEARS)

## Load the aligned table

`processed/aligned.parquet` is the reconciled output: every market re-classified,
matched to a game (moneyline + spread), with price snapshots, Elo, travel, and
betting-line features joined where available. If any feature table is missing the
join is skipped, so this notebook works after a partial pipeline run.

In [ ]:
a = pd.read_parquet(config.PROCESSED_DIR / 'aligned.parquet')
print(f'aligned: {len(a):,} rows, {len(a.columns)} cols')
print('feature cols:', [c for c in a.columns if c.startswith(('elo_','travel_','timezone','sportsbook','open_','price_delta','spread_','n_','snapshot'))])

## 1. Market population by type and season

In [ ]:
a['year'] = a['market_end'].astype(str).str[:4]
type_year = a.pivot_table(index='year', columns='market_type',
                          values='market_id', aggfunc='count', fill_value=0)
type_year.loc['Total'] = type_year.sum()
type_year

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
type_counts = a['market_type'].value_counts()
ax.barh(type_counts.index[::-1], type_counts.values[::-1])
ax.set_xlabel('Number of resolved markets')
ax.set_title('Resolved NFL markets by type (re-classified)')
plt.tight_layout();

## 2. Classifier health

Sanity-check the re-classified distribution. The previous classifier collapsed
real game spreads into `moneyline` (they carry two team-name outcomes) and
mixed game totals, season win totals, and team totals into one `totals` bucket.
The fix splits these into `spread`, `game_total`, `season_total`, `player_props`,
etc. The modeling targets are `moneyline` and `spread`.

In [ ]:
print(a['market_type'].value_counts())
print()
print('Modeling targets (matched to a game):')
for mt in ['moneyline', 'spread']:
    sub = a[a['market_type'] == mt]
    n_matched = (sub['match_status'] == 'matched').sum()
    print(f'  {mt:10s}: {len(sub):5d} markets, {n_matched:5d} matched ({n_matched/len(sub)*100:.0f}%)')

## 3. Price-history coverage + open→close delta

For each market we snapshot the last hourly close at or before T-1h (the modeling
label baseline) and the first traded hour's open (the `price_delta` feature).

In [ ]:
matched = a[a['match_status'] == 'matched']
print(f'matched markets: {len(matched):,}')
print(f'  with a snapshot price:    {matched["snapshot_price"].notna().sum():,}')
print(f'  with an open price:       {matched["open_price"].notna().sum():,}')
print(f'  with a price_delta:       {matched["price_delta"].notna().sum():,}')
print(f'  clean cutoff snapshots:   {(matched["snapshot_method"]=="cutoff").sum():,}')
print()
print('candles per matched market (coverage diagnostic):')
print(matched['n_candles'].describe())

## 4. Match rate (moneyline + spread → NFL game)

In [ ]:
print('Match status (all markets):')
print(a['match_status'].value_counts())
print()
for mt in ['moneyline', 'spread']:
    sub = a[a['market_type'] == mt]
    print(f'{mt}: match_status')
    print(sub['match_status'].value_counts())
    print()

## 5. Feature coverage on matched markets

Elo (computed from nflverse scores, 63-68% outcome accuracy), travel/timezone
(from a bundled team-locations table), and sportsbook betting lines (Kaggle).

In [ ]:
feats = ['elo_home_pre', 'elo_away_pre', 'elo_home_prob',
         'travel_distance_km', 'timezone_shift_h',
         'sportsbook_spread', 'sportsbook_ou',
         'open_price', 'price_delta', 'snapshot_price']
cov = pd.DataFrame({
    'non_null': matched[feats].notna().sum(),
    'coverage': matched[feats].notna().mean(),
})
print(cov)

## 6. Calibration sanity

Group matched moneyline markets into bins by their pre-game snapshot price
(implied probability) and compare to the realized win rate. A well-calibrated
market sits near the diagonal. This is a market check, not a model check.

In [ ]:
clean = a[
    (a['market_type'] == 'moneyline')
    & (a['match_status'] == 'matched')
    & (a['snapshot_method'] == 'cutoff')
    & a['snapshot_price'].notna()
    & a['resolved_yes_price'].notna()
].copy()
print(f'moneyline markets usable for calibration: {len(clean)}')
if len(clean) >= 20:
    clean['prob_bin'] = pd.cut(clean['snapshot_price'], bins=np.linspace(0, 1, 11))
    cal = clean.groupby('prob_bin', observed=True).agg(
        n=('resolved_yes_price', 'size'),
        realized=('resolved_yes_price', 'mean'),
        implied=('snapshot_price', 'mean'),
    )
    print(cal)

In [ ]:
if len(clean) >= 20:
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='perfect calibration')
    ax.scatter(cal['implied'], cal['realized'], s=cal['n'] * 3, alpha=0.7)
    ax.set_xlabel('Implied probability (snapshot price)')
    ax.set_ylabel('Realized win rate')
    ax.set_title('Polymarket calibration (market check)')
    ax.legend()
    plt.tight_layout();

## Summary

Fill in the headline numbers once the pipeline has run end-to-end:

| Metric | Value |
|---|---|
| Resolved NFL markets (total) | _ |
| Moneyline markets (re-classified) | _ |
| Spread markets (re-classified) | _ |
| Moneylines matched to a game | _ |
| Spreads matched to a game | _ |
| Matched w/ clean cutoff snapshot + label | _ |
| Matched w/ Elo features | _ |
| Matched w/ sportsbook line | _ |

**Decision gate:** the clean-snapshot × matched count now spans both moneylines
and spreads. Elo outcome accuracy should land ~63-68% (a correctly-implemented
team-Elo baseline). With spreads included the labeled row count roughly doubles,
which improves the viability of a direct price-prediction / CLV approach.